# Rebulding Transformer

In this code, I'm trying to rebuild all components inside the Transformer framework. The main reason is to actually learn what's really happening behind this framework, the tricks to make some algorithm faster, and also just to fill my free time. I will try my best to include this file with my own explanation for each part of the code. 

References: <br>
Vaswani et al. (2017) "Attention Is All You Need", https://arxiv.org/pdf/1706.03762 <br>
Rush & Klein (2018) "The Annotated Transformer" (Harvard NLP), https://nlp.seas.harvard.edu/annotated-transformer

### Import libraries

In [ ]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad, softmax
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

# Set to False to skip notebook execution (e.g. for debugging)
warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

#### Helper Function

In [71]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)

# Attention Mechanism

## Scaled Dot-Product Attention

We have matrices $Q$, $K$, and $V$. Based on Gemini explanation, we can think of this as something like this. Let's say we want to watch some youtube videos and we go to search bar

$Q$: The text that we type on search bar. (What we are looking for)<br>
$K$: The titles and tags of all videos. (What the system matches my search against)<br>
$V$: The actual videos that you watch. (The information I receive once match is found)

As stated in the paper, the mathematical function can be written as: 

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The way I understand this function, is first we check the "similarity" between $Q$ and $K$ using a dot product. Next, we scale it because if we don't the dot product can grow very large in magnitude and it will push softmax function to regions where it has extremely small gradients. After that, the softmax will transform this to a range of possibilities, where the total probabilities is 1. Next, we are doing another dot product with $V$. The last part may be interpreted as how much value we you want to pass. 

### More detailed explanation on why we scaled it

Let's say that we want to do a dot product of $Q$ and $K$, both with dimension $d_k$, and the components has mean 0 and variance 1. The result of the dot product is below: 

$$Q \cdot K = \sum_{i=1}^{d_k} q_i k_i$$

If we apply on what we learn in probabilities & statistic course here, the dot product will have zero mean with variance $d_k$. This means, if $d_k$ grows large, the variance of the dot product will grow as well. If we let this variance grow, this will affect the softmax function as we have some value that are too small and too big due to very high variance. In some extreme cases, let's say we have these value [100, -100, 10]. The softmax will make the highest value to be super close to 1 while others close to 0 (saturated). This will introduce us with vanishing gradient problem. The function below is the derivative of the softmax function. 

$$\frac{\partial S_i}{\partial z_j} = S_i (\delta_{ij} - S_j)$$

where

$$\delta_{ij} = \begin{cases} 
1 & \text{if } i = j \\ 
0 & \text{if } i \neq j 
\end{cases}$$

According to the function above, consider these 4 possibilities
* $i \neq j$ and $S_i \to 1$
    * The equation will become $-S_i S_j$. Since $S_i$ approach 1, $S_j$ will approach zero. This will make derivative approach zero.

* $i \neq j$ and $S_i \to 0$
    * Since $S_i$ approach 0, then the derivative will approach zero as well.

* $i = j$ and $S_i \to 1$
    * The equation will become $S_i(1-S_i)$. The $(1-S_i)$ term will approach zero. This will make derivative approach zero.

* $i = j$ and $S_i \to 0$
    * Since $S_i$ approach 0, then the derivative will approach zero as well.

Based on those 4 possibilities, it's clear that all will lead to derivative approaching zero, which is bad. To counter this, if we divide the dot product by $\sqrt{d_k}$, this will allow the dot product variance to be zero, which means that it will not pose a major problem to softmax function and later to the gradient function. 

### Building intuition of 3D matrix dot product

Let's say I have two different matrix, matrix A with dimension (Ax, Ay, Az) and matrix B with dimension (Bx, By, Bz). We want to do multiplication of A·B. We can assume that Ax = Bx. 

3D matrix itself can be visualised as "stack" of 2D matrices. In order to do dot product on two different 3D matrix, we will multiply corresponding 2D matrices across the "stack". 

This is only possible if Az = By, and the resultant matrix will be in the form of (Ax, Ay, Bz). 

Example: Multiplying a stack of ten (3x2) matrices with a stack of ten (2x4) matrices. This operation will result in ten (3x4) matrices.

The work inside this multiplication
- 1st stack of the resultant is obtained by multiplying the first stack in (3x2) matrices collection with the first stack in (2x4) collections
- 2nd stack of the resultant is obtained by multiplying the second stack in (3x2) matrices collection with the second stack in (2x4) collections
- continue form 3rd stack to 10th stack

In [72]:
def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Computing the scaled dot-product attention. 

    Input shape:
        query: (batch_size, seq_len, d_k)
        key:   (batch_size, seq_len, d_k)
        value: (batch_size, seq_len, d_v)
    """
    d_k = query.size(-1)
    
    transposed_K = torch.transpose(key, -1, -2)
    scores = torch.matmul(query, transposed_K)
    scaled_scores =  scores / math.sqrt(d_k)

    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask==0, -1e9)

    attention_weights = softmax(scaled_scores, dim = -1)
    if dropout is not None:
        attention_weights = dropout(attention_weights)

    output = torch.matmul(attention_weights, value)

    return output, attention_weights

## Multi-Head Attention

Using a single attention head would not be enough since we will need to "averaging" everything into one thing. For example, you can't have one head that focuses on grammar and other head that focuses on vocabulary since you only have one head. You are forced to mix this all together into one. Just imagine how it is if you eat a Nasi Padang but you combine everything, so every bite you will only get the same taste. Keeping it separate will allow the cuisine and each of the condiment to really show how good they are. 

It's better if we use multi-head attention, where each head will focus on different things. The formula can be seen below. 

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$
$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

What we are doing here, is to project query, key, and value h times with different and learned linear projections to $d_k$, $d_k$, $d_v$ dimensions. Then, we will continue by performing attention in parallel. The output then will be concatenated and will be projected back to $d_{model}$. 

If you see the code below, you can see that we don't have $W_1$ or $W_2$ inside the projection. What actually happen is that we "combine" all of $W_i$ into one matrix. This is also the reason why the linear layers projects from d_model to d_model. This is done to use the fast GPU matrix multplication algorithm to make the model runs faster.

The usage of view and transpose here is quite important as well. The view is the fast O(1) way for us to reshaping the matrix, with the possibility of making it non-contigous (non-sequential). The transpose is basically just "swapping" the two dimension there. The reason we calculate it like this is to call the attention function only once, since all the data has been combined together. 

Once we are done, we combine it back together. We use contigous() here is to "force" the matrix to be contigous again by changing its memory. The next steps are just reshaping and projecting it back to d_model. 

In [73]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % h == 0, "d_model must be divisible by h"
        
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h # d_v = d_k
        
        # linear layers
        self.w_q = nn.Linear(d_model, d_model) 
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_out = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        """
        Input shape:
            query: tensor of shape [batch_size, seq_len, d_model]
            key:   tensor of shape [batch_size, seq_len, d_model]
            value: tensor of shape [batch_size, seq_len, d_model]
            mask:  tensor of shape [batch_size, 1, seq_len, seq_len]
        """
        nbatches = query.size(0)
        
        # projected tensor, output shape of each: [batch_size, seq_len, d_model]
        q_projected = self.w_q(query)
        k_projected = self.w_k(key)
        v_projected = self.w_v(value)

        # divide model dimension to h heads
        q_heads = q_projected.view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        k_heads = k_projected.view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        v_heads = v_projected.view(nbatches, -1, self.h, self.d_k).transpose(1, 2)

        attn_output, weights = scaled_dot_product_attention(q_heads, k_heads, v_heads, mask, self.dropout)

        # combine it back
        concat_output = attn_output.transpose(1, 2).contiguous().view(nbatches, -1, self.d_model)
        
        return self.w_out(concat_output)

## Position-wise Feed-Forward Networks

In [74]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        
        # two linear layers
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Input shape:
            x: tensor of shape [batch_size, seq_len, d_model]
        """
        
        return self.w_2(self.dropout(self.w_1(x).relu()))

## Positional Encoding

In [75]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # changing the term a bit for numerical stability
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        
        # Register pe as a buffer
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Input shape:
            x: Input token embedding tensor of shape [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, : x.size(1)]
        
        return self.dropout(x)

## Sublayer Connection

I'm using the Pre-LN version because the one written inside the paper (Post-LN) has some issues that may cause gradient to decay/compound exponentially. 

In [76]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [77]:
class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_fn):
        """
        Args:
            x: Input tensor from the previous layer [batch_size, seq_len, d_model]
            sublayer_fn: A lambda function or module (like your Attention or FFN)
        """
        return x + self.dropout(sublayer_fn(self.norm(x)))

## Encoder Layer

In [78]:
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        """
        Args:
            x: Input tensor of shape [batch_size, seq_len, d_model]
            mask: Attention mask tensor
        """
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, lambda x: self.feed_forward(x))

In [79]:
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        """
        Pass the input (and mask) through each layer in the stack sequentially.
        """
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)

## Decoder

In [80]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        
        # We need 3 sublayer connections now!
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        """
        Args:
            x: Target sequence tokens [batch_size, tgt_len, d_model]
            memory: The final output vectors from the Encoder [batch_size, src_len, d_model]
            src_mask: Mask to hide padding tokens in the encoder source
            tgt_mask: Causal mask to hide future tokens in the decoder target
        """
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, memory, memory, src_mask))
    
        return self.sublayer[2](x, lambda x: self.feed_forward(x))

In [81]:
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        """
        Args:
            x: Target sequence tokens
            memory: Final output from the Encoder
            src_mask: Padding mask for the encoder memory
            tgt_mask: Causal look-ahead mask for the decoder target
        """
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
            
        return self.norm(x)

## Encoder-Decoder

In [82]:
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed  # Token Embeddings + Positional Encoding
        self.tgt_embed = tgt_embed  # Token Embeddings + Positional Encoding
        self.generator = generator  # Final Linear layer + Softmax to project to vocabulary

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        return self.decode(memory, src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [83]:
class Generator(nn.Module):
    """Define standard linear + softmax generation step."""
    def __init__(self, d_model, vocab_size):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

In [84]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [85]:
def make_model(
    src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1
):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadAttention(d_model, h)
    ff = PositionWiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

In [86]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.uint8
    )
    return subsequent_mask == 0

In [90]:
def inference_test():
    test_model = make_model(11, 11, 2)
    test_model.eval()
    src = torch.LongTensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
    src_mask = torch.ones(1, 1, 10)

    memory = test_model.encode(src, src_mask)
    ys = torch.zeros(1, 1).type_as(src)

    for i in range(9):
        out = test_model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = test_model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )

    print("Example Untrained Model Prediction:", ys)


def run_tests():
    for _ in range(10):
        inference_test()


show_example(run_tests)

Example Untrained Model Prediction: tensor([[0, 0, 0, 8, 2, 5, 9, 5, 9, 5]])
Example Untrained Model Prediction: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Example Untrained Model Prediction: tensor([[0, 7, 6, 7, 6, 7, 6, 7, 6, 7]])
Example Untrained Model Prediction: tensor([[0, 2, 2, 2, 2, 2, 9, 9, 9, 6]])
Example Untrained Model Prediction: tensor([[ 0,  9,  6,  9,  6, 10,  4,  4,  4,  4]])
Example Untrained Model Prediction: tensor([[ 0, 10,  2,  0,  4,  0,  0,  0,  0,  0]])
Example Untrained Model Prediction: tensor([[0, 1, 0, 2, 9, 1, 0, 2, 9, 1]])
Example Untrained Model Prediction: tensor([[0, 9, 0, 6, 7, 0, 6, 7, 0, 6]])
Example Untrained Model Prediction: tensor([[0, 6, 1, 7, 5, 4, 4, 4, 4, 4]])
Example Untrained Model Prediction: tensor([[0, 9, 9, 9, 9, 9, 9, 9, 9, 9]])
